# 🔤 Module 3 — Class 3 Homework: Categorical Encoding

**Topic:** Encoding — get_dummies, OneHotEncoder, binary mapping

## 📋 What you have to do

This notebook **already has working code**. Your job:

1. **Run every cell from top to bottom** — it should run with no errors.
2. **Add a comment on EVERY code cell** that explains what the cell does, in your own words.
   - Use `#` to write comments inside the code cell.
   - Write at least 1 short sentence per cell. Longer is better.
3. **Save your notebook** as `Module<X>_Class<Y>_<YourName>.ipynb` and submit.

**Example of a good commented cell:**

```python
# Count how many customers churned vs stayed
# value_counts() returns the number of rows for each unique value
df['Churn'].value_counts()
```

**Example of a BAD comment (do not do this):**

```python
# count
df['Churn'].value_counts()
```

---


In [ ]:
# === SETUP — run this first ===
# Import essential libraries for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Silence warnings to keep the notebook clean
import warnings; warnings.filterwarnings('ignore')

# URL of the dataset (IBM Telco Customer Churn)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

# Load the dataset from the URL into a DataFrame
df = pd.read_csv(url)

# Convert 'TotalCharges' column to numeric, fixing blank spaces with 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Print the final shape of the loaded data
print('Loaded:', df.shape)

Loaded: (7043, 21)


### Cell 1 — see which columns are categorical

In [ ]:
# Extract names of all columns containing text/categorical data
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Print the list of identified categorical columns
print('Categorical columns:', cat_cols)

# Print the total count of these columns
print('Total:', len(cat_cols))

Categorical columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
Total: 17


### Cell 2 — look at one categorical column

In [ ]:
df['Contract'].value_counts() #Contact column values count

### Cell 3 — use pd.get_dummies to one-hot encode Contract

In [ ]:
# Convert the 'Contract' column into binary (0 or 1) dummy columns
dummies = pd.get_dummies(df['Contract'], prefix='Contract')

# Compare the shape of the data before and after the transformation
print('Shape before:', df[['Contract']].shape, '-> after:', dummies.shape)

# Preview the first few rows of the newly created dummy DataFrame
dummies.head()

Shape before: (7043, 1) -> after: (7043, 3)


,Contract_Month-to-month,Contract_One year,Contract_Two year
0,True,False,False
1,False,True,False
2,True,False,False
3,False,True,False
4,True,False,False


### Cell 4 — drop_first=True (to avoid the dummy variable trap)

In [ ]:
# Create dummy variables but drop the first category column to avoid redundancy
dummies2 = pd.get_dummies(df['Contract'], prefix='Contract', drop_first=True)

# Print the new shape to see that we have one less column than before
print('With drop_first:', dummies2.shape)

# Preview the first few rows of the optimized dummy DataFrame
dummies2.head()

With drop_first: (7043, 2)


,Contract_One year,Contract_Two year
0,False,False
1,True,False
2,False,False
3,True,False
4,False,False


### Cell 5 — use OneHotEncoder from sklearn

In [ ]:
# Import the OneHotEncoder transformer from sklearn
from sklearn.preprocessing import OneHotEncoder

# Initialize the encoder to output a standard dense NumPy array
ohe = OneHotEncoder(sparse_output=False)

# Fit and transform the 'PaymentMethod' column into a binary array
encoded = ohe.fit_transform(df[['PaymentMethod']])

# Convert the array to a DataFrame, getting actual category names for columns
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['PaymentMethod']))

# Preview the first few rows of the one-hot encoded DataFrame
encoded_df.head()

,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0


### Cell 6 — binary Yes/No → 1/0 mapping

In [ ]:
# Map 'No' to 0 and 'Yes' to 1 to create a binary target column
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})

# Preview both columns together to verify the mapping worked correctly
df[['Churn', 'Churn_bin']].head()

,Churn,Churn_bin
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


### Cell 7 — encode multiple columns at once


In [ ]:
# Convert multiple categorical columns to dummy variables at once, dropping the first category
multi = pd.get_dummies(df[['gender', 'Contract', 'PaymentMethod']], drop_first=True)

# Print the shape of the new combined DataFrame to see the total number of columns
print('Shape:', multi.shape)

# Preview the first few rows of the final encoded DataFrame
multi.head()

Shape: (7043, 6)


,gender_Male,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,False,False,False,False,True,False
1,True,True,False,False,False,True
2,True,False,False,False,False,True
3,True,True,False,False,False,False
4,False,False,False,False,True,False
